In [1]:
import os
if os.getcwd().endswith("notebooks"):
    os.chdir("..")

import json

In [2]:
import pandas as pd
manifest = pd.read_csv("data/processed/manifest.csv")
match_id = manifest["match_id"].iloc[0]   # just take the first one

with open(f"data/raw/{match_id}_timeline.json") as f:
    tl = json.load(f)

frames = tl["info"]["frames"]
print(len(frames), "frames")

32 frames


In [3]:
types = set()
for frame in frames:
    for e in frame["events"]:
        types.add(e["type"])
print(types)

{'ELITE_MONSTER_KILL', 'WARD_PLACED', 'WARD_KILL', 'ITEM_DESTROYED', 'ITEM_PURCHASED', 'DRAGON_SOUL_GIVEN', 'OBJECTIVE_BOUNTY_PRESTART', 'ITEM_UNDO', 'LEVEL_UP', 'CHAMPION_KILL', 'TURRET_PLATE_DESTROYED', 'ITEM_SOLD', 'OBJECTIVE_BOUNTY_FINISH', 'CHAMPION_SPECIAL_KILL', 'PAUSE_END', 'SKILL_LEVEL_UP', 'BUILDING_KILL', 'GAME_END'}


In [4]:
seen = set()
for frame in frames:
    for e in frame["events"]:
        if e["type"] in ("ELITE_MONSTER_KILL", "BUILDING_KILL") and e["type"] not in seen:
            print(e["type"], "->", e)
            seen.add(e["type"])

ELITE_MONSTER_KILL -> {'bounty': 0, 'killerId': 2, 'killerTeamId': 100, 'monsterType': 'HORDE', 'position': {'x': 4713, 'y': 10030}, 'timestamp': 520292, 'type': 'ELITE_MONSTER_KILL'}
BUILDING_KILL -> {'bounty': 0, 'buildingType': 'TOWER_BUILDING', 'killerId': 9, 'laneType': 'BOT_LANE', 'position': {'x': 10504, 'y': 1029}, 'teamId': 100, 'timestamp': 600056, 'towerType': 'OUTER_TURRET', 'type': 'BUILDING_KILL'}


In [5]:
import pandas as pd, json
manifest = pd.read_csv("data/processed/manifest.csv")

plate_times = []
for mid in manifest["match_id"].head(50):
    try:
        with open(f"data/raw/{mid}_timeline.json") as f:
            tl = json.load(f)
    except FileNotFoundError:
        continue
    for frame in tl["info"]["frames"]:
        for e in frame["events"]:
            if e["type"] == "TURRET_PLATE_DESTROYED":
                plate_times.append(e["timestamp"] / 60000)

print(len(plate_times), "plate events")
print("max minute:", max(plate_times) if plate_times else None)
print(pd.Series(plate_times).describe())

2333 plate events
max minute: 43.672216666666664
count    2333.000000
mean       17.560780
std         7.539447
min         2.306283
25%        12.295233
50%        16.934783
75%        22.730850
max        43.672217
dtype: float64


In [11]:
import pandas as pd
df = pd.read_csv("data/processed/features.csv")

print(df.head(10))
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print(df.describe())
print(df.isna().sum())

        match_id  minute  gold_diff  xp_diff  level_diff  cs_diff  tower_diff  dragon_diff  baron_diff  herald_diff  grub_diff  won
0  KR_8261949619       0          0        0           0        0           0            0           0            0          0    0
1  KR_8261949619       1       -520      -26           0        0           0            0           0            0          0    0
2  KR_8261949619       2       -654      -93           0       -8           0            0           0            0          0    0
3  KR_8261949619       3        -67     -338           0      -25           0            0           0            0          0    0
4  KR_8261949619       4       -426     -553          -1      -35           0            0           0            0          0    0
5  KR_8261949619       5       -225     -339          -2      -39           0            0           0            0          0    0
6  KR_8261949619       6      -2014    -1400          -3      -62           

In [7]:
# late-game rows should show clear separation
late = df[df["minute"] >= 20]
print(late.groupby("won")[["gold_diff", "xp_diff", "tower_diff", "dragon_diff"]].mean())

       gold_diff      xp_diff  tower_diff  dragon_diff
won                                                   
0   -3824.144491 -5047.579030   -1.649061    -1.208074
1    4124.511358  4520.092943    2.000983     0.378013


In [8]:
print(df["dragon_diff"].describe())
print(df["dragon_diff"].value_counts().sort_index())

count    128699.000000
mean         -0.228922
std           1.280937
min          -6.000000
25%          -1.000000
50%           0.000000
75%           0.000000
max           5.000000
Name: dragon_diff, dtype: float64
dragon_diff
-6        2
-5       95
-4     1362
-3     4790
-2    12377
-1    24500
 0    59221
 1    16380
 2     6870
 3     2471
 4      591
 5       40
Name: count, dtype: int64


In [12]:
import json, pandas as pd
manifest = pd.read_csv("data/processed/manifest.csv")

missing = 0
total = 0
killer_zero = 0
for mid in manifest["match_id"].head(300):
    try:
        with open(f"data/raw/{mid}_timeline.json") as f:
            tl = json.load(f)
    except FileNotFoundError:
        continue
    for frame in tl["info"]["frames"]:
        for e in frame["events"]:
            if e["type"] == "ELITE_MONSTER_KILL":
                total += 1
                if "killerTeamId" not in e:
                    missing += 1
                if e.get("killerId") == 0:
                    killer_zero += 1

print(f"{total} elite monster kills, {missing} missing killerTeamId, {killer_zero} with killerId 0")

2305 elite monster kills, 0 missing killerTeamId, 16 with killerId 0


In [13]:
for c in ["tower_diff", "dragon_diff", "baron_diff", "herald_diff", "grub_diff"]:
    print(c, round(df[c].mean(), 4))

tower_diff 0.0515
dragon_diff -0.2289
baron_diff -0.0036
herald_diff 0.0654
grub_diff 0.365
